# LRU-2Q Parameter Sweep (a1in_fraction)

Line graphs of hit rate and promotions vs. `a1in_fraction` for each preset.

- **x-axis**: `a1in_fraction` (fraction of hot capacity reserved for A1in)
- **y-axis**: hit rate (%) or promotions count
- **one line per preset**

CSV source: `../lru2q_sweep.csv` generated by `scripts/bench_lru2q_sweep.sh`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

CSV_PATH = Path('../lru2q_sweep.csv')
df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(df)} rows from {CSV_PATH}')
print(df.columns.tolist())

In [ ]:
# Derived + canonical ordering
df['hit_pct'] = df['hit_rate'] * 100

PRESETS = ['steady_state', 'frequency_favored', 'recency_favored', 'high_churn', 'hot_set']
PRESET_LABELS = {
    'steady_state':      'Steady State',
    'frequency_favored': 'Freq Favored',
    'recency_favored':   'Recency Favored',
    'high_churn':        'High Churn',
    'hot_set':           'Hot Set',
}

# Ensure a1in_fraction is treated as numeric and sorted
df['a1in_fraction'] = df['a1in_fraction'].astype(float)
df = df.sort_values(['a1in_fraction', 'preset', 'run']).reset_index(drop=True)

# Average over runs if RUNS > 1
agg = (
    df.groupby(['preset', 'a1in_fraction'])
      .agg(
          hit_pct   = ('hit_pct', 'mean'),
          promotions= ('promotions', 'mean'),
          demotions = ('demotions', 'mean'),
      )
      .reset_index()
)

agg.head()

## Hit Rate vs. `a1in_fraction`

Each line is a preset; y-axis is hit rate (%). This shows how sensitive 2Q is to
the size of its probationary A1in buffer under different workloads.

In [ ]:
plt.figure(figsize=(10, 6))

colors = {
    'steady_state':      '#4C72B0',
    'frequency_favored': '#C44E52',
    'recency_favored':   '#55A868',
    'high_churn':        '#8172B2',
    'hot_set':           '#E07B39',
}

for preset in PRESETS:
    sub = agg[agg['preset'] == preset].sort_values('a1in_fraction')
    if sub.empty:
        continue
    plt.plot(
        sub['a1in_fraction'], sub['hit_pct'],
        marker='o', linewidth=2, markersize=5,
        label=PRESET_LABELS[preset], color=colors.get(preset, None),
    )

plt.xlabel('a1in_fraction (probationary A1in byte fraction)', fontsize=11)
plt.ylabel('Hit Rate (%)', fontsize=11)
plt.title('LRU-2Q Hit Rate vs. a1in_fraction', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3, linestyle='--')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('lru2q_hit_rate_vs_a1in.png', dpi=150, bbox_inches='tight')
plt.show()

## Promotions vs. `a1in_fraction`

Each line is a preset; y-axis is the number of promotions (cold→hot moves) during
the measurement window. Higher promotions generally mean more churn and more I/O
cost; combined with the hit-rate plot, you can see where extra promotions are
actually buying better placement vs. just thrashing.

In [ ]:
plt.figure(figsize=(10, 6))

for preset in PRESETS:
    sub = agg[agg['preset'] == preset].sort_values('a1in_fraction')
    if sub.empty:
        continue
    plt.plot(
        sub['a1in_fraction'], sub['promotions'],
        marker='s', linewidth=2, markersize=5,
        label=PRESET_LABELS[preset], color=colors.get(preset, None),
    )

plt.xlabel('a1in_fraction (probationary A1in byte fraction)', fontsize=11)
plt.ylabel('Promotions (count)', fontsize=11)
plt.title('LRU-2Q Promotions vs. a1in_fraction', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3, linestyle='--')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('lru2q_promotions_vs_a1in.png', dpi=150, bbox_inches='tight')
plt.show()

## Optional: Demotions vs. `a1in_fraction`

Cold writes are the expensive operations in a cloud tiering system (S3 PUTs). This
plot shows how demotions change as A1in grows/shrinks.

In [ ]:
plt.figure(figsize=(10, 6))

for preset in PRESETS:
    sub = agg[agg['preset'] == preset].sort_values('a1in_fraction')
    if sub.empty:
        continue
    plt.plot(
        sub['a1in_fraction'], sub['demotions'],
        marker='^', linewidth=2, markersize=5,
        label=PRESET_LABELS[preset], color=colors.get(preset, None),
    )

plt.xlabel('a1in_fraction (probationary A1in byte fraction)', fontsize=11)
plt.ylabel('Demotions (count)', fontsize=11)
plt.title('LRU-2Q Demotions vs. a1in_fraction', fontsize=13, fontweight='bold')
plt.grid(alpha=0.3, linestyle='--')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('lru2q_demotions_vs_a1in.png', dpi=150, bbox_inches='tight')
plt.show()